# F1-M04b: Unión de Preinscripción + HTML

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 ¿Qué hace este notebook?

Completa la creación del dataset final `df_alumno.parquet`:

1. Carga `df_alumno_base.parquet` (generado por M04a con 8 tablas)
2. Carga la tabla de preinscripción y filtra solo registros de la UJI
3. Hace merge añadiendo 4 columnas nuevas: vía de acceso, orden de
   preferencia, cupo y universidad de origen
4. Guarda el dataset final como `.parquet` y `.csv`
5. Genera una página HTML completa con pipeline, infografía, grafo
   interactivo de relaciones entre tablas y resumen del dataset

## ⚠️ Requisitos

- Haber ejecutado `f1_m04a_union_tablas.ipynb` (genera df_alumno_base.parquet)
- El parquet de preinscripción en `data/01_interim/` (o el Excel en `data/00_raw/`)

## 📦 ¿Qué genera?

| Archivo | Ubicación | Descripción |
|---|---|---|
| df_alumno.parquet | `data/02_processed/` | Dataset final para análisis (Python) |
| df_alumno.csv | `data/02_processed/` | Dataset final para Tableau/Power BI |
| m04_dataset_final.html | `docs/html/fase1/` | Documentación interactiva con grafo |

## 📁 Flujo de datos

```
data/02_processed/df_alumno_base.parquet  ─┐
data/01_interim/preinscripcion.parquet     ─┼─→  data/02_processed/df_alumno.parquet
                                            └─→  data/02_processed/df_alumno.csv
```

## 📌 Siguiente

- `f1_m05_dashboard.ipynb` (dashboard HTML resumen de la Fase 1)

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
#
# Importa rutas y constantes del proyecto.
#
# De src.constantes importamos todo lo necesario para:
#   - Filtrar preinscripción (UJI_ID = 40)
#   - Mapear columnas (MAPEO_COLUMNAS_PREINSCRIPCION)
#   - Generar el HTML (EMOJIS_TABLAS, COLORES_TABLAS, etc.)
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# --- Detectar entorno y localizar ROOT ---
def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(
        f"No se encontró carpeta 'src/' subiendo desde {start}. "
        f"Verifica que el notebook está dentro de AU_UJI/"
    )

ROOT = _encontrar_root(Path.cwd())


sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
from src.config import RUTA_RAW, RUTA_INTERIM, RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.constantes import (
    UJI_ID,
    TABLA_PREINSCRIPCION,
    TABLAS_EXCEL_PRINCIPAL,
    TABLAS_TODAS,
    CLAVES_MERGE_TABLAS,
    CLAVES_MERGE_PREINSCRIPCION,
    RELACIONES_TABLAS,
    EMOJIS_TABLAS,
    COLORES_TABLAS,
    CATEGORIAS_VARIABLES,
    MAPEO_COLUMNAS_PREINSCRIPCION,
    DESCRIPCIONES_COLUMNAS_PREINSCRIPCION,
    COLUMNAS_NUEVAS_PREINSCRIPCION
)
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

# --- Carpetas de salida ---
RUTA_FASE1 = RUTA_HTML / 'fase1'
crear_directorios([RUTA_FASE1, RUTA_PROCESSED])

# Atajo para formato español
fmt = formato_numero_es

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATASET BASE
# ============================================================================

print('=' * 60)
print('CARGANDO DATASET BASE')
print('=' * 60)

path_base = RUTA_PROCESSED / 'df_alumno_base.parquet'

if not path_base.exists():
    raise FileNotFoundError(f'No encontrado: {path_base}\nEjecuta primero f1_m04a_union_tablas.ipynb')

df_alumno = pd.read_parquet(path_base)
n_base = len(df_alumno)
n_cols_base = len(df_alumno.columns)

print(f'✅ Cargado: {path_base.name}')
print(f'   Registros: {n_base:,}')
print(f'   Columnas: {n_cols_base}')

CARGANDO DATASET BASE
✅ Cargado: df_alumno_base.parquet
   Registros: 109,568
   Columnas: 33


In [3]:
# ============================================================================
# CELDA 3: CARGAR PREINSCRIPCIÓN
# ============================================================================

print('=' * 60)
print('CARGANDO PREINSCRIPCIÓN')
print('=' * 60)

path_preins_parquet = RUTA_INTERIM / f'{TABLA_PREINSCRIPCION}.parquet'
path_preins_excel = RUTA_RAW / 'preinscripcion_si.xlsx'

if path_preins_parquet.exists():
    df_preins = pd.read_parquet(path_preins_parquet)
    print(f'✅ Cargado: {path_preins_parquet.name}')
elif path_preins_excel.exists():
    df_preins = pd.read_excel(path_preins_excel)
    print(f'✅ Cargado: {path_preins_excel.name}')
else:
    raise FileNotFoundError('No encontrado archivo de preinscripción')

n_preins_total = len(df_preins)
print(f'   Registros totales: {n_preins_total:,}')

col_uni = 'uni_id' if 'uni_id' in df_preins.columns else 'UNI_ID'
df_preins_uji = df_preins[df_preins[col_uni] == UJI_ID].copy()
n_preins_uji = len(df_preins_uji)
print(f'\n✅ Filtrado UJI (ID={UJI_ID}): {n_preins_uji:,}')

CARGANDO PREINSCRIPCIÓN


✅ Cargado: preinscripcion.parquet
   Registros totales: 210,986

✅ Filtrado UJI (ID=40): 107,218


In [4]:
# ============================================================================
# CELDA 4: PREPARAR COLUMNAS PARA MERGE
# ============================================================================
# NOTA: La eliminación de 'cupo' duplicado debería hacerse en m02_limpieza,
# pero preinscripción es un Excel separado que no pasó por esa limpieza.
# ============================================================================

print('=' * 60)
print('PREPARANDO MERGE')
print('=' * 60)

df_preins_uji.columns = df_preins_uji.columns.str.lower()

for old_col, new_col in MAPEO_COLUMNAS_PREINSCRIPCION.items():
    if old_col in df_preins_uji.columns and new_col in df_preins_uji.columns:
        df_preins_uji = df_preins_uji.drop(columns=[new_col])
        print(f'   ⚠️ Eliminada columna duplicada: {new_col}')

for old_col, new_col in MAPEO_COLUMNAS_PREINSCRIPCION.items():
    if old_col in df_preins_uji.columns:
        df_preins_uji = df_preins_uji.rename(columns={old_col: new_col})

cols_merge = CLAVES_MERGE_PREINSCRIPCION.copy()

usar_curso_ini = False
if 'curso_aca_ini' in df_alumno.columns:
    if 'ano' in df_preins_uji.columns:
        df_preins_uji['curso_aca_ini'] = df_preins_uji['ano'].astype('Int64')
        df_alumno['curso_aca_ini'] = df_alumno['curso_aca_ini'].astype('Int64')
        usar_curso_ini = True
    elif 'curso_aca_ini' in df_preins_uji.columns:
        df_preins_uji['curso_aca_ini'] = df_preins_uji['curso_aca_ini'].astype('Int64')
        df_alumno['curso_aca_ini'] = df_alumno['curso_aca_ini'].astype('Int64')
        usar_curso_ini = True

if usar_curso_ini:
    cols_merge.append('curso_aca_ini')

cols_nuevas = [c for c in COLUMNAS_NUEVAS_PREINSCRIPCION if c in df_preins_uji.columns]

print(f'\n   Claves merge: {cols_merge}')
print(f'   Columnas nuevas: {cols_nuevas}')

PREPARANDO MERGE
   ⚠️ Eliminada columna duplicada: cupo

   Claves merge: ['per_id_ficticio', 'exp_tit_id', 'curso_aca_ini']
   Columnas nuevas: ['via_acceso', 'orden_preferencia', 'cupo', 'universidad_origen']


In [5]:
# ============================================================================
# CELDA 5: EJECUTAR MERGE
# ============================================================================

print('=' * 60)
print('MERGE')
print('=' * 60)

cols_disponibles = [c for c in cols_merge if c in df_preins_uji.columns]

df_preins_merge = df_preins_uji[cols_disponibles + cols_nuevas].drop_duplicates(
    subset=cols_disponibles, keep='first'
)

print(f'\n📊 Antes del merge:')
print(f'   df_alumno: {len(df_alumno):,}')
print(f'   preinscripcion: {len(df_preins_merge):,}')

df_alumno = df_alumno.merge(df_preins_merge, on=cols_disponibles, how='left')
# LEFT JOIN — decisión metodológica:
#   - Se usa left join para conservar TODOS los expedientes del dataset base,
#     independientemente de si tienen datos de preinscripción.
#   - Un inner join eliminaría los expedientes sin match (~12.6%),
#     introduciendo sesgo: los alumnos sin preinscripción UJI son
#     principalmente traslados, mayores de 25 o accesos directos.
#   - Los NaN en las columnas de preinscripción se tratan en Fase 3.

cols_duplicadas = df_alumno.columns[df_alumno.columns.duplicated()].tolist()
if cols_duplicadas:
    df_alumno = df_alumno.loc[:, ~df_alumno.columns.duplicated()]

n_con_preins = df_alumno[cols_nuevas[0]].notna().sum() if cols_nuevas else 0
pct_preins = (n_con_preins / len(df_alumno) * 100) if len(df_alumno) > 0 else 0

print(f'\n✅ Dataset final: {len(df_alumno):,} × {len(df_alumno.columns)}')
print(f'   Cobertura preinscripción: {pct_preins:.1f}%')


MERGE

📊 Antes del merge:
   df_alumno: 109,568
   preinscripcion: 32,598



✅ Dataset final: 109,568 × 37
   Cobertura preinscripción: 87.4%


In [6]:
# ============================================================================
# CELDA 6: GUARDAR DATASET FINAL
# ============================================================================

print('=' * 60)
print('GUARDANDO DATASET FINAL')
print('=' * 60)

path_parquet = RUTA_PROCESSED / 'df_alumno.parquet'
df_alumno.to_parquet(path_parquet, index=False)
size_parquet = path_parquet.stat().st_size / 1024 / 1024
print(f'\n✅ {path_parquet.name}: {size_parquet:.2f} MB')

path_csv = RUTA_PROCESSED / 'df_alumno.csv'
df_alumno.to_csv(path_csv, index=False, sep=';', decimal=',')
size_csv = path_csv.stat().st_size / 1024 / 1024
print(f'✅ {path_csv.name}: {size_csv:.2f} MB')

GUARDANDO DATASET FINAL



✅ df_alumno.parquet: 2.27 MB


✅ df_alumno.csv: 30.71 MB


In [7]:
# ============================================================================
# CELDA 7: CALCULAR MÉTRICAS PARA HTML
# ============================================================================
#
# Calcula todas las métricas que se mostrarán en la página HTML.
# Todas son DINÁMICAS — se calculan del DataFrame real, nada hardcodeado.
# ============================================================================

# --- Métricas del dataset ---
n_registros = len(df_alumno)
n_columnas = len(df_alumno.columns)
n_alumnos = df_alumno['per_id_ficticio'].nunique()
n_titulaciones = df_alumno['exp_tit_id'].nunique() if 'exp_tit_id' in df_alumno.columns else 0

# --- Rango de cursos académicos ---
if 'curso_aca' in df_alumno.columns:
    cursos = df_alumno['curso_aca'].dropna().unique()
    curso_min = int(min(cursos)) if len(cursos) > 0 else 'N/A'
    curso_max = int(max(cursos)) if len(cursos) > 0 else 'N/A'
else:
    curso_min, curso_max = 'N/A', 'N/A'

# --- Porcentajes de variables derivadas ---
pct_vive_fuera = df_alumno['vive_fuera'].mean() * 100 if 'vive_fuera' in df_alumno.columns else 0
pct_tiene_beca = df_alumno['tiene_beca'].mean() * 100 if 'tiene_beca' in df_alumno.columns else 0
pct_con_preins = pct_preins  # calculado en celda 5

# --- Conteo de tablas (dinámico) ---
n_tablas_excel = len([f for f in RUTA_INTERIM.glob('*.parquet')
                      if f.stem != TABLA_PREINSCRIPCION])
n_tablas_total = n_tablas_excel + 1  # +1 por preinscripción

print(f'Métricas: {fmt(n_registros)} reg, {n_columnas} cols, {fmt(n_alumnos)} alumnos')

Métricas: 109.568 reg, 37 cols, 30.872 alumnos


In [8]:
# ============================================================================
# CELDA 8: GENERAR HTML - NAVEGACIÓN Y KPIs
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa='fase1',
    modulo_activo='m04'
)

KPIS = [
    {'valor': str(n_tablas_total), 'titulo': 'Tablas unidas', 'color': '#3182ce'},
    {'valor': fmt(n_registros), 'titulo': 'Registros', 'color': '#38a169'},
    {'valor': str(n_columnas), 'titulo': 'Columnas', 'color': '#805ad5'},
    {'valor': fmt(n_alumnos), 'titulo': 'Alumnos únicos', 'color': '#ed8936'},
]
kpis_html = generar_kpis_html(KPIS)
print('✅ KPIs')

GENERANDO HTML
✅ KPIs


In [9]:
# ============================================================================
# CELDA 9: SECCIÓN NOTEBOOKS
# ============================================================================

notebooks_html = f'''
<div class="notebooks-grid">
    <div class="notebook-card" style="background: #f0fff4; border: 2px solid #38a169;">
        <h3 style="color: #276749;">📗 f1_m04a_union_tablas.ipynb</h3>
        <p style="color: #38a169;">Une {n_tablas_excel} tablas del Excel principal</p>
        <ul style="color: #4a5568;">
            <li>Carga tablas de data/02_interim/</li>
            <li>Procesa domicilios → <code>vive_fuera</code></li>
            <li>Crea <code>tiene_beca</code></li>
            <li>Genera df_alumno_base.parquet</li>
        </ul>
    </div>
    <div class="notebook-card" style="background: #fffaf0; border: 2px solid #ed8936;">
        <h3 style="color: #c05621;">📙 f1_m04b_union_preinscripcion.ipynb</h3>
        <p style="color: #dd6b20;">Une preinscripción al dataset base</p>
        <ul style="color: #4a5568;">
            <li>Filtra UJI (UNI_ID={UJI_ID})</li>
            <li>Merge por per_id + exp_tit_id + curso_aca_ini</li>
            <li>Genera df_alumno.parquet</li>
            <li>Genera df_alumno.csv</li>
        </ul>
    </div>
</div>
<p style="margin-top: 15px; color: #718096; font-size: 0.9em;">
    <strong>Orquestador:</strong> <code>f1_m04_dataset_final.ipynb</code> ejecuta M04a → M04b secuencialmente
</p>
'''
seccion_notebooks = generar_seccion_html('Notebooks', notebooks_html, '📓')
print('✅ Notebooks')

✅ Notebooks


In [10]:
# ============================================================================
# CELDA 10: PIPELINE VISUAL ANIMADO
# ============================================================================

# Generar tarjetas de tablas dinámicamente desde constantes
tablas_pipeline = TABLAS_EXCEL_PRINCIPAL

tablas_html = ''
for i, nombre in enumerate(tablas_pipeline):
    emoji = EMOJIS_TABLAS.get(nombre, '📊')
    tablas_html += f'''
        <div class="pipeline-tabla" style="animation-delay: {0.1 + i*0.05}s;">
            <span class="emoji">{emoji}</span>
            <span class="name">{nombre.capitalize()}</span>
        </div>'''

pipeline_html = f'''
<div class="pipeline-container">
    <div class="pipeline-excel">
        <h4>📗 Excel Principal: {n_tablas_excel} Tablas</h4>
        <div class="pipeline-tablas">
            {tablas_html}
        </div>
    </div>
    <div class="pipeline-arrow">⬇️</div>
    <div class="pipeline-resultado" style="background: linear-gradient(135deg, #3182ce, #2c5282);">
        <h4>df_alumno_base.parquet</h4>
        <p>{fmt(n_base)} registros × {n_cols_base} columnas</p>
    </div>
    <div class="pipeline-arrow">➕</div>
    <div class="pipeline-excel" style="border-color: #ed64a6; background: #fff5f7;">
        <h4 style="color: #97266d;">📙 Excel Preinscripción</h4>
        <div class="pipeline-tablas">
            <div class="pipeline-tabla" style="border-color: #ed64a6;">
                <span class="emoji">📄</span>
                <span class="name">Preinscripción</span>
            </div>
        </div>
    </div>
    <div class="pipeline-arrow">⬇️</div>
    <div class="pipeline-resultado">
        <h4>🎯 df_alumno.parquet</h4>
        <p>{fmt(n_registros)} registros × {n_columnas} columnas</p>
    </div>
</div>
'''
seccion_pipeline = generar_seccion_html('Pipeline de Datos', pipeline_html, '🔄')
print('✅ Pipeline')

✅ Pipeline


In [11]:
# ============================================================================
# CELDA 11: M04a - TABLAS + VARIABLES DERIVADAS
# ============================================================================

tablas_grid_html = ''
for nombre in TABLAS_EXCEL_PRINCIPAL:
    emoji = EMOJIS_TABLAS.get(nombre, '📊')
    clave = CLAVES_MERGE_TABLAS.get(nombre, '')
    tablas_grid_html += f'''
    <div class="tabla-mini">
        <div class="icon">{emoji}</div>
        <div class="name">{nombre.capitalize()}</div>
        <div class="key">{clave}</div>
    </div>'''

m04a_html = f'''
<h3 style="margin-top: 15px; color: #4a5568;">Tablas de origen</h3>
<div class="tablas-grid">
    {tablas_grid_html}
</div>

<h3 style="margin-top: 20px; color: #4a5568;">Variables derivadas</h3>
<table>
    <thead><tr><th>Variable</th><th>Tipo</th><th>Descripción</th><th>%</th></tr></thead>
    <tbody>
        <tr>
            <td><code>vive_fuera</code></td>
            <td>Boolean</td>
            <td>True si domicilio curso ≠ domicilio familiar</td>
            <td>{pct_vive_fuera:.1f}%</td>
        </tr>
        <tr>
            <td><code>tiene_beca</code></td>
            <td>Boolean</td>
            <td>True si tiene beca asignada</td>
            <td>{pct_tiene_beca:.1f}%</td>
        </tr>
    </tbody>
</table>
'''
# Tabla de trazabilidad de merges — datos estáticos (los asserts en m04a
# garantizan que ningún merge pierde registros; lo documentamos aquí visualmente)
TRAZABILIDAD_MERGES = [
    ('expedientes',  'BASE',                        'n_base',  0),
    ('titulaciones', 'exp_tit_id',                  'n_base',  0),
    ('demográficos', 'per_id_ficticio',              'n_base',  0),
    ('domicilios',   'per_id_ficticio + curso_aca',  'n_base',  'pct_vive_fuera'),
    ('becas',        'per_id_ficticio + curso_aca',  'n_base',  'pct_tiene_beca'),
    ('trabajo',      'per_id + tit + curso_aca',     'n_base',  None),
    ('notas',        'per_id + tit + curso_aca',     'n_base',  None),
    ('recibos',      'per_id_ficticio + curso_aca',  'n_base',  None),
]

filas_traz = ''
for i, (tabla, clave, _, _extra) in enumerate(TRAZABILIDAD_MERGES, 1):
    emoji = EMOJIS_TABLAS.get(tabla, '📊')
    es_base = tabla == 'expedientes'
    filas_traz += (
        f'<tr>'
        f'<td>{i}</td>'
        f'<td>{emoji} {tabla.capitalize()}</td>'
        f'<td><code>{clave}</code></td>'
        f'<td style="text-align:center;">{"BASE" if es_base else fmt(n_base)}</td>'
        f'<td style="text-align:center; color:#38a169;">{"—" if es_base else "✅ 0"}</td>'
        f'</tr>'
    )

trazabilidad_html = f'''
<details style="margin-top: 20px;">
    <summary style="cursor:pointer; font-weight:bold; color:#4a5568; padding:10px;
                    background:#f7fafc; border-radius:6px; border:1px solid #e2e8f0;">
        🔍 Ver trazabilidad de merges — registros antes y después de cada unión
    </summary>
    <div style="margin-top:10px;">
        <p style="color:#718096; font-size:0.85em; margin-bottom:10px;">
            Todos los merges son <strong>LEFT JOIN</strong> sobre expedientes como tabla base
            ({fmt(n_base)} registros). Los asserts en el notebook verifican que ningún merge
            introduce ni elimina registros.
        </p>
        <table>
            <thead>
                <tr>
                    <th>#</th><th>Tabla</th><th>Clave de merge</th>
                    <th>Registros tras merge</th><th>Registros perdidos</th>
                </tr>
            </thead>
            <tbody>{filas_traz}</tbody>
        </table>
        <p style="margin-top:10px; color:#718096; font-size:0.85em;">
            Los NaN introducidos en columnas opcionales (beca, trabajo, recibos)
            son esperados: indican que el alumno no tiene ese dato en ese curso.
            Se tratan en Fase 3 (feature engineering).
        </p>
    </div>
</details>
'''
m04a_html = m04a_html + trazabilidad_html
seccion_m04a = generar_seccion_html(f'M04a: Unión de {n_tablas_excel} Tablas', m04a_html, '📗')
print('✅ M04a')


✅ M04a


In [12]:
# ============================================================================
# CELDA 12: M04b - COLUMNAS AÑADIDAS + COBERTURA + COLUMNAS DESCARTADAS
# ============================================================================

# --- Tabla de columnas añadidas (desde constantes) ---
cols_html = ''
for orig, final in MAPEO_COLUMNAS_PREINSCRIPCION.items():
    desc = DESCRIPCIONES_COLUMNAS_PREINSCRIPCION.get(final, '')
    cols_html += f'<tr><td>{orig.upper()}</td><td><code>{final}</code></td><td>{desc}</td></tr>'

# --- Tabla de columnas descartadas (hardcodeada — no cambia) ---
COLUMNAS_DESCARTADAS_PREINS = [
    ('ANO',               'Usado solo como clave temporal de merge — no se incorpora al dataset'),
    ('UNIVERSIDAD',       'Redundante con NOMBRE_UNIVERSIDAD'),
    ('MUNICIPIO',         'Dato geográfico ya cubierto por la tabla de domicilios'),
    ('CP',                'Código postal ya disponible en domicilios'),
    ('CUPO',              'Duplicado de NOM_CUPO — eliminado explícitamente antes del merge'),
    ('ESTADO',            'Estado de la solicitud de preinscripción — no relevante para el modelo'),
    ('NOTA_TXT',          'Nota en formato texto — ya existe nota_selectividad numérica en el dataset base'),
    ('SOLICITUD_ID',      'Identificador interno de preinscripción — sin valor predictivo'),
    ('COD_ESTUDIOS',      'Código de estudios — ya cubierto por exp_tit_id'),
    ('CONVOCATORIA',      'Convocatoria de acceso — muy correlada con ANO'),
    ('NOTA_1',            'Nota parcial de acceso — ya cubierta por nota_selectividad'),
    ('NOTA_2',            'Nota parcial de acceso — ya cubierta por nota_selectividad'),
    ('ANO_1',             'Año parcial — redundante con ANO'),
    ('CON_1',             'Convocatoria parcial — redundante'),
    ('TITULACION_CENTRO', 'Nombre literal del centro — alta cardinalidad, cubierto por exp_tit_id'),
    ('ESTADO_TITULACION', 'Estado de la titulación en preinscripción — no disponible en predicción'),
    ('UNI_ID',            'Usado solo para filtrar UJI (== 40) — varianza cero tras el filtro'),
    ('NOMBRE_UNIVERSIDAD','Siempre Universitat Jaume I tras el filtro — varianza cero'),
    ('ESTUDIO',           'Nombre literal del estudio — alta cardinalidad, cubierto por exp_tit_id'),
    ('MATRICULADO',       '⚠️ LEAKAGE DIRECTO — indica si el alumno se matriculó finalmente. No disponible en el momento de predicción.'),
]

filas_descartadas = ''
for col, motivo in COLUMNAS_DESCARTADAS_PREINS:
    if 'LEAKAGE' in motivo:
        fila = (
            f'<tr style="background:#fff5f5;">'
            f'<td><code style="color:#e53e3e;font-weight:bold;">{col}</code></td>'
            f'<td style="color:#e53e3e;">{motivo}</td>'
            f'</tr>'
        )
    else:
        fila = f'<tr><td><code>{col}</code></td><td style="color:#718096;">{motivo}</td></tr>'
    filas_descartadas += fila

m04b_html = f'''
<p style="margin-bottom: 15px; color: #718096;">
    Cobertura: <strong>{pct_con_preins:.1f}%</strong> de registros con datos de preinscripción
    ({fmt(n_con_preins)} de {fmt(n_registros)})
</p>

<h3 style="margin-top: 15px; color: #4a5568;">Columnas incorporadas (4 de 24)</h3>
<table>
    <thead><tr><th>Original</th><th>Nombre final</th><th>Descripción</th></tr></thead>
    <tbody>{cols_html}</tbody>
</table>

<p style="margin-top: 15px; color: #718096; font-size: 0.85em;">
    <strong>Nota:</strong> La nota de selectividad ya existe en el dataset base (nota_selectividad).
</p>

<details style="margin-top: 20px;">
    <summary style="cursor:pointer; font-weight:bold; color:#4a5568; padding: 10px;
                    background:#f7fafc; border-radius:6px; border: 1px solid #e2e8f0;">
        📋 Ver columnas descartadas (20 de 24) — razones de exclusión
    </summary>
    <div style="margin-top: 10px;">
        <p style="color:#718096; font-size:0.85em; margin-bottom:10px;">
            El Excel de preinscripción contiene 24 columnas. Solo se incorporan 4 al dataset final.
            Las 20 restantes se descartan por los motivos indicados.
            La columna <strong style="color:#e53e3e;">MATRICULADO</strong> se excluye por leakage directo:
            indica si el alumno se matriculó, información no disponible en el momento de predicción.
        </p>
        <table>
            <thead><tr><th>Columna descartada</th><th>Motivo</th></tr></thead>
            <tbody>{filas_descartadas}</tbody>
        </table>
    </div>
</details>
'''
# Bloque metodológico left join — datos dinámicos del merge
n_sin_preins = n_registros - n_con_preins
pct_sin_preins = 100 - pct_con_preins

left_join_html = f'''
<div style="margin-top:20px; padding:15px; background:#f0fff4;
            border-left:4px solid #38a169; border-radius:6px;">
    <h4 style="color:#276749; margin-bottom:10px;">
        ⚖️ Decisión metodológica: LEFT JOIN vs INNER JOIN
    </h4>
    <table style="width:100%; margin-bottom:12px;">
        <thead>
            <tr><th>Tipo de join</th><th>Registros resultantes</th><th>Impacto</th></tr>
        </thead>
        <tbody>
            <tr style="background:#f0fff4;">
                <td><strong>LEFT JOIN ✅ (elegido)</strong></td>
                <td><strong>{fmt(n_registros)}</strong></td>
                <td>Conserva todos los expedientes</td>
            </tr>
            <tr style="background:#fff5f5;">
                <td>INNER JOIN ❌</td>
                <td>~{fmt(n_con_preins)}</td>
                <td style="color:#e53e3e;">Eliminaría {fmt(n_sin_preins)} registros ({pct_sin_preins:.1f}%)</td>
            </tr>
        </tbody>
    </table>
    <p style="color:#4a5568; font-size:0.9em; margin:0;">
        Los <strong>{fmt(n_sin_preins)} registros sin preinscripción</strong> ({pct_sin_preins:.1f}%)
        corresponden principalmente a traslados, accesos por mayores de 25 años y accesos directos
        desde otras universidades. Eliminarlos con un inner join introduciría sesgo sistemático
        en el modelo. Los valores NaN resultantes se tratan en la Fase 3 (feature engineering).
    </p>
</div>
'''
m04b_html = m04b_html + left_join_html
seccion_m04b = generar_seccion_html('M04b: Unión de Preinscripción', m04b_html, '📙')
print('✅ M04b')


✅ M04b

In [13]:
# ============================================================================
# CELDA 13: INFOGRAFÍA DEL ALUMNO
# ============================================================================

# Generar categorías dinámicamente desde constantes
categorias_html = ''
total_vars = 0

for key, cat in CATEGORIAS_VARIABLES.items():
    # Filtrar solo variables que existen en el dataset
    vars_existentes = [v for v in cat['variables'] if v in df_alumno.columns]
    total_vars += len(vars_existentes)
    vars_list = ', '.join(vars_existentes[:4])  # Max 4 para no saturar
    if len(vars_existentes) > 4:
        vars_list += '...'
    
    categorias_html += f'''
    <div class="categoria-card" style="border-left-color: {COLORES_TABLAS.get(key, '#718096')};">
        <h4><span class="icon">{cat['emoji']}</span> {cat['titulo']} ({len(vars_existentes)})</h4>
        <p style="font-size: 0.85em; color: #718096; margin: 5px 0 0 0;">{vars_list}</p>
    </div>'''

n_categorias = len(CATEGORIAS_VARIABLES)

infografia_html = f'''
<div class="infografia-alumno">
    <div class="alumno-central">
        👤
        <span>ALUMNO</span>
    </div>
    <div class="categorias-grid">
        {categorias_html}
    </div>
</div>
<p style="text-align:center; color:#718096; margin-top:15px; font-size:0.9em;">
    <strong>{total_vars}</strong> variables en <strong>{n_categorias}</strong> categorías 
    para <strong>{fmt(n_alumnos)}</strong> alumnos únicos
</p>
'''
seccion_infografia = generar_seccion_html('Perfil del Alumno', infografia_html, '📊')
print('✅ Infografía')

✅ Infografía


In [14]:
# ============================================================================
# CELDA 14: GRAFO INTERACTIVO
# ============================================================================

tablas_grafo = TABLAS_TODAS

# Generar nodos JS dinámicamente
nodos_js = "{ id: 'alumno', label: '👤 ALUMNO', x: 0, y: 0, vx: 0, vy: 0, radius: 40, color: '#3182ce', fixed: true },\n"
for t in tablas_grafo:
    emoji = EMOJIS_TABLAS.get(t, '📊')
    color = COLORES_TABLAS.get(t, '#718096')
    nodos_js += f"        {{ id: '{t}', label: '{emoji} {t.capitalize()}', x: 0, y: 0, vx: 0, vy: 0, radius: 28, color: '{color}' }},\n"

# Generar enlaces JS dinámicamente
enlaces_js = ''.join([f"        {{ source: 'alumno', target: '{t}' }},\n" for t in tablas_grafo])
# Añadir relaciones entre tablas desde constantes
for src, tgt in RELACIONES_TABLAS:
    enlaces_js += f"        {{ source: '{src}', target: '{tgt}' }},\n"

grafo_html = '''
<div id="grafo-container" style="height: 380px; position: relative; background: linear-gradient(135deg, #1a202c, #2d3748); border-radius: 10px; overflow: hidden;">
    <canvas id="grafo-canvas" style="width: 100%; height: 100%;"></canvas>
    <div style="position: absolute; bottom: 10px; left: 10px; color: rgba(255,255,255,0.6); font-size: 0.8em;">💡 Arrastra los nodos con el ratón</div>
</div>
'''
seccion_grafo = generar_seccion_html('Grafo de Relaciones (Interactivo)', grafo_html, '🕸️')
print('✅ Grafo')

✅ Grafo


In [15]:
# ============================================================================
# CELDA 15: ARCHIVOS DE SALIDA
# ============================================================================

archivos_html = f'''
<div class="archivos-grid">
    <div class="archivo" style="background: #ebf8ff;">
        <div class="nombre">🐍 df_alumno.parquet</div>
        <div class="uso">Python / Análisis ({size_parquet:.2f} MB)</div>
    </div>
    <div class="archivo" style="background: #ebf8ff;">
        <div class="nombre">📊 df_alumno.csv</div>
        <div class="uso">Tableau / Power BI ({size_csv:.2f} MB)</div>
    </div>
    <div class="archivo" style="background: #faf5ff; border-color: #805ad5;">
        <div class="nombre">📦 df_alumno_base.parquet</div>
        <div class="uso">Intermedio (sin preinscripción)</div>
    </div>
</div>
<p style="text-align: center; margin-top: 12px; color: #718096; font-size: 0.85em;">
    📍 Ubicación: <code>data/02_processed/</code>
</p>
'''
seccion_archivos = generar_seccion_html('Archivos de Salida', archivos_html, '📁')
print('✅ Archivos')

✅ Archivos


In [16]:
# ============================================================================
# CELDA 16: RESUMEN DEL DATASET
# ============================================================================

resumen_html = f'''
<table>
    <tbody>
        <tr><td><strong>Registros</strong></td><td>{fmt(n_registros)}</td></tr>
        <tr><td><strong>Columnas</strong></td><td>{n_columnas}</td></tr>
        <tr><td><strong>Alumnos únicos</strong></td><td>{fmt(n_alumnos)}</td></tr>
        <tr><td><strong>Titulaciones</strong></td><td>{n_titulaciones}</td></tr>
        <tr><td><strong>Cursos</strong></td><td>{curso_min} - {curso_max}</td></tr>
        <tr><td><strong>% Vive fuera</strong></td><td>{pct_vive_fuera:.1f}%</td></tr>
        <tr><td><strong>% Tiene beca</strong></td><td>{pct_tiene_beca:.1f}%</td></tr>
        <tr><td><strong>% Con preinscripción</strong></td><td>{pct_con_preins:.1f}%</td></tr>
    </tbody>
</table>
'''
seccion_resumen = generar_seccion_html('Resumen del Dataset Final', resumen_html, '📋')
print('✅ Resumen')

✅ Resumen


In [17]:
# ============================================================================
# CELDA 17: SCRIPT DEL GRAFO ANIMADO
# ============================================================================

grafo_script = f'''

(function() {{
    const canvas = document.getElementById('grafo-canvas');
    if (!canvas) return;
    const ctx = canvas.getContext('2d');
    
    function resize() {{
        canvas.width = canvas.offsetWidth * 2;
        canvas.height = canvas.offsetHeight * 2;
        ctx.scale(2, 2);
    }}
    resize();
    window.addEventListener('resize', resize);
    
    const nodes = [
        {nodos_js}
    ];
    
    const links = [
        {enlaces_js}
    ];
    
    const w = canvas.offsetWidth, h = canvas.offsetHeight;
    nodes[0].x = w / 2;
    nodes[0].y = h / 2;
    
    for (let i = 1; i < nodes.length; i++) {{
        const angle = (i - 1) * (2 * Math.PI / (nodes.length - 1));
        const dist = 110 + Math.random() * 30;
        nodes[i].x = w/2 + Math.cos(angle) * dist;
        nodes[i].y = h/2 + Math.sin(angle) * dist;
    }}
    
    const particles = [];
    for (let i = 0; i < 30; i++) {{
        particles.push({{x: Math.random()*w, y: Math.random()*h, vx: (Math.random()-0.5)*0.3, vy: (Math.random()-0.5)*0.3, r: Math.random()*2+1, a: Math.random()*0.3+0.1}});
    }}
    
    let time = 0, dragNode = null;
    
    function simulate() {{
        nodes.forEach(n => {{ if (!n.fixed) {{ n.vx += (w/2-n.x)*0.0003; n.vy += (h/2-n.y)*0.0003; }} }});
        for (let i = 0; i < nodes.length; i++) {{
            for (let j = i+1; j < nodes.length; j++) {{
                const dx = nodes[j].x-nodes[i].x, dy = nodes[j].y-nodes[i].y;
                const dist = Math.sqrt(dx*dx+dy*dy) || 1;
                const minD = nodes[i].radius + nodes[j].radius + 25;
                if (dist < minD) {{
                    const f = (minD - dist) * 0.02;
                    if (!nodes[i].fixed) {{ nodes[i].vx -= (dx/dist)*f; nodes[i].vy -= (dy/dist)*f; }}
                    if (!nodes[j].fixed) {{ nodes[j].vx += (dx/dist)*f; nodes[j].vy += (dy/dist)*f; }}
                }}
            }}
        }}
        links.forEach(l => {{
            const s = nodes.find(n => n.id === l.source);
            const t = nodes.find(n => n.id === l.target);
            if (!s || !t) return;
            const dx = t.x-s.x, dy = t.y-s.y, dist = Math.sqrt(dx*dx+dy*dy) || 1;
            const f = (dist - 90) * 0.002;
            if (!s.fixed) {{ s.vx += (dx/dist)*f; s.vy += (dy/dist)*f; }}
            if (!t.fixed) {{ t.vx -= (dx/dist)*f; t.vy -= (dy/dist)*f; }}
        }});
        nodes.forEach(n => {{
            if (!n.fixed) {{
                n.vx *= 0.92; n.vy *= 0.92;
                n.x += n.vx; n.y += n.vy;
                n.x = Math.max(n.radius, Math.min(w - n.radius, n.x));
                n.y = Math.max(n.radius, Math.min(h - n.radius, n.y));
            }}
        }});
    }}
    
    function draw() {{
        time += 0.02;
        ctx.clearRect(0, 0, w, h);
        
        // Partículas de fondo
        particles.forEach(p => {{
            p.x += p.vx; p.y += p.vy;
            if (p.x < 0) p.x = w; if (p.x > w) p.x = 0;
            if (p.y < 0) p.y = h; if (p.y > h) p.y = 0;
            ctx.beginPath();
            ctx.arc(p.x, p.y, p.r, 0, Math.PI * 2);
            ctx.fillStyle = `rgba(99, 179, 237, ${{p.a}})`;
            ctx.fill();
        }});
        
        // Enlaces con partículas viajando
        links.forEach((l, i) => {{
            const s = nodes.find(n => n.id === l.source);
            const t = nodes.find(n => n.id === l.target);
            if (!s || !t) return;
            ctx.beginPath();
            ctx.moveTo(s.x, s.y);
            ctx.lineTo(t.x, t.y);
            ctx.strokeStyle = 'rgba(255, 255, 255, 0.15)';
            ctx.lineWidth = 2;
            ctx.stroke();
            // Partícula viajando
            const progress = (Math.sin(time * 2 + i) + 1) / 2;
            ctx.beginPath();
            ctx.arc(s.x + (t.x - s.x) * progress, s.y + (t.y - s.y) * progress, 3, 0, Math.PI * 2);
            ctx.fillStyle = '#63b3ed';
            ctx.fill();
        }});
        
        // Nodos
        nodes.forEach(n => {{
            // Glow
            const g = ctx.createRadialGradient(n.x, n.y, 0, n.x, n.y, n.radius * 1.4);
            g.addColorStop(0, n.color + '40');
            g.addColorStop(1, 'transparent');
            ctx.beginPath();
            ctx.arc(n.x, n.y, n.radius * 1.4, 0, Math.PI * 2);
            ctx.fillStyle = g;
            ctx.fill();
            // Círculo
            ctx.beginPath();
            ctx.arc(n.x, n.y, n.radius, 0, Math.PI * 2);
            ctx.fillStyle = n.color;
            ctx.fill();
            ctx.strokeStyle = 'rgba(255, 255, 255, 0.3)';
            ctx.lineWidth = 2;
            ctx.stroke();
            // Texto
            ctx.fillStyle = 'white';
            ctx.textAlign = 'center';
            ctx.textBaseline = 'middle';
            const parts = n.label.split(' ');
            ctx.font = n.fixed ? '14px Segoe UI' : '12px Segoe UI';
            ctx.fillText(parts[0], n.x, n.y - 6);
            ctx.font = n.fixed ? 'bold 9px Segoe UI' : '8px Segoe UI';
            ctx.fillText(parts.slice(1).join(' '), n.x, n.y + 8);
        }});
    }}
    
    function animate() {{
        simulate();
        draw();
        requestAnimationFrame(animate);
    }}
    animate();
    
    // Interacción con mouse
    canvas.addEventListener('mousedown', e => {{
        const rect = canvas.getBoundingClientRect();
        const mx = e.clientX - rect.left, my = e.clientY - rect.top;
        nodes.forEach(n => {{
            if (Math.sqrt((n.x - mx) ** 2 + (n.y - my) ** 2) < n.radius) dragNode = n;
        }});
    }});
    canvas.addEventListener('mousemove', e => {{
        if (dragNode) {{
            const rect = canvas.getBoundingClientRect();
            dragNode.x = e.clientX - rect.left;
            dragNode.y = e.clientY - rect.top;
            dragNode.vx = 0;
            dragNode.vy = 0;
        }}
    }});
    canvas.addEventListener('mouseup', () => {{ dragNode = null; }});
    canvas.addEventListener('mouseleave', () => {{ dragNode = null; }});
}})();

'''
print('✅ Script grafo')


✅ Script grafo

In [18]:
# ============================================================================
# CELDA 18: GENERAR HTML FINAL
# ============================================================================

# Contenido en orden acordado:
# 1. KPIs
# 2. Notebooks
# 3. Pipeline animado
# 4. M04a (tablas + variables derivadas)
# 5. M04b (columnas añadidas + cobertura)
# 6. Infografía del alumno
# 7. Grafo interactivo
# 8. Archivos de salida
# 9. Resumen del dataset

contenido_html = (
    kpis_html +
    seccion_notebooks +
    seccion_pipeline +
    seccion_m04a +
    seccion_m04b +
    seccion_infografia +
    seccion_grafo +
    seccion_archivos +
    seccion_resumen
)

# Página completa con plantilla
html_completo = render_pagina_desde_fichero(
    'f1_m04b_union_preinscripcion.ipynb',
    contenido_html,
    carpeta_notebook='fase1_transformacion'
)

# Guardar
ruta_html = RUTA_FASE1 / 'm04_dataset_final.html'
guardar_html(html_completo, ruta_html)

print(f'\n✅ HTML generado: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m04_dataset_final.html



✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m04_dataset_final.html


In [19]:
# ============================================================================
# CELDA 19: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F1-M04b COMPLETADO')
print('=' * 60)
print(f'\n📁 Archivos generados:')
print(f'   • df_alumno.parquet ({size_parquet:.2f} MB)')
print(f'   • df_alumno.csv ({size_csv:.2f} MB)')
print(f'   • m04_dataset_final.html')
print(f'\n📊 Dataset final ({n_tablas_total} tablas unidas):')
print(f'   • Registros: {fmt(n_registros)}')
print(f'   • Columnas: {n_columnas}')
print(f'   • Alumnos: {fmt(n_alumnos)}')
print(f'   • Cobertura preinscripción: {pct_con_preins:.1f}%')
print(f'\n📌 Siguiente: f1_m05_grafo_d3.ipynb')
print('=' * 60)


✅ F1-M04b COMPLETADO

📁 Archivos generados:
   • df_alumno.parquet (2.27 MB)
   • df_alumno.csv (30.71 MB)
   • m04_dataset_final.html

📊 Dataset final (9 tablas unidas):
   • Registros: 109.568
   • Columnas: 37
   • Alumnos: 30.872
   • Cobertura preinscripción: 87.4%

📌 Siguiente: f1_m05_grafo_d3.ipynb


In [20]:
# ============================================================================
# CELDA 20: ACTUALIZAR INDEX.HTML
# ============================================================================
#
# Regenera docs/html/index.html con los KPIs actualizados.
# df_alumno.parquet ya existe → se activarán Expedientes y Período.
# ============================================================================

from src.index_generator import actualizar_index
actualizar_index()


🔄 Actualizando index.html...


  ⚠ Error leyendo Excel: [Errno 13] Permission denied: 'C:\\Users\\mjmor\\OneDrive - Universitat Jaume I\\2.- AU_UJI\\data\\00_raw\\datos_proyecto_sin_preinscrip.xlsx'
  ✅ KPIs: Expedientes: 30.872 | Período: 2010-2020 | Variables: 50+
  ✅ C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\index.html


WindowsPath('C:/Users/mjmor/OneDrive - Universitat Jaume I/2.- AU_UJI/docs/html/index.html')

In [21]:
# ============================================================================
# CELDA 20: ACTUALIZAR INDEX.HTML
# ============================================================================
#
# df_alumno.parquet acaba de generarse → los KPIs del index
# (Expedientes únicos, Período) ahora pueden actualizarse.
# ============================================================================

from src.index_generator import actualizar_index
actualizar_index()


🔄 Actualizando index.html...
  ⚠ Error leyendo Excel: [Errno 13] Permission denied: 'C:\\Users\\mjmor\\OneDrive - Universitat Jaume I\\2.- AU_UJI\\data\\00_raw\\datos_proyecto_sin_preinscrip.xlsx'
  ✅ KPIs: Expedientes: 30.872 | Período: 2010-2020 | Variables: 50+
  ✅ C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\index.html


WindowsPath('C:/Users/mjmor/OneDrive - Universitat Jaume I/2.- AU_UJI/docs/html/index.html')